# Stocker — End-to-end Colab workflow

Self-contained: clones the repo, installs deps, builds the dataset, loads
**`google/gemma-4-E4B-it`** in-process via `transformers` (4-bit BnB),
pre-caches all 7 specialist votes, runs **GRPO** on the moderator LoRA via
TRL, and saves loss / reward plots + the trained adapter.

Designed for a free **Colab T4** (16 GB VRAM) — no separate vLLM server
needed. If you have an L4/A100, drop `load_in_4bit=False` for full bf16.

Outputs live under `training/runs/<timestamp>/`:
- `moderator-lora/` — trained PEFT adapter
- `loss.png`, `reward.png` — training curves
- `eval_pre.json`, `eval_post.json` — pre/post backtest reports

> Tip: `Runtime → Change runtime type → T4 GPU` before running.

In [1]:
# Install — keep Colab's pre-baked pandas/numpy (TF + cudf depend on them);
# only upgrade what we strictly need.
#
# Big upgrades (transformers/trl/peft) are intentional — Colab ships old.
# Everything else uses --upgrade-strategy only-if-needed so we don't
# break google-colab/tensorflow/gradio/etc.
!pip install -q -U 'transformers>=4.55' 'trl>=0.11' 'peft>=0.13' 'accelerate>=1.0' 'bitsandbytes>=0.43' 'datasets>=3.0'
!pip install -q --upgrade-strategy=only-if-needed yfinance mplfinance pyarrow 'pydantic>=2' pydantic-settings 'openai>=1' tensorboard 'huggingface_hub>=1.10'
# Indicators are computed in-repo (app/data/indicators.py) — no pandas-ta.

In [ ]:
import os, sys, pathlib

# 1) Already in a stocker repo? (running from VSCode on a local clone, or
#    the repo was manually uploaded to Colab). Detect and skip the clone.
def _is_stocker_root(p: str) -> bool:
    return os.path.isfile(os.path.join(p, "app", "council", "specialists.py"))

CANDIDATES = [os.getcwd(), "/content/stocker", "/workspace/stocker"]
WORKDIR = next((c for c in CANDIDATES if _is_stocker_root(c)), None)

if WORKDIR is None:
    # 2) Not present — clone. EDIT THIS URL before running on a fresh Colab.
    REPO_URL = "https://github.com/CRIMSONHydra/stocker.git"
    WORKDIR  = "/content/stocker"
    assert "<your-username>" not in REPO_URL, (
        "Edit REPO_URL in this cell to your fork before running on a fresh runtime."
    )
    !git clone {REPO_URL} {WORKDIR}
    assert _is_stocker_root(WORKDIR), f"Clone failed — {WORKDIR}/app/council/ missing."

os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print("Working dir:", WORKDIR)

Working dir: /content/stocker


In [3]:
# Gemma 4 is Apache-2.0 (not gated) — token just lifts download rate limits.
from huggingface_hub import login
import getpass
login(token=getpass.getpass("HF token (read scope is enough): "))

In [ ]:
# Build the bundled dataset (yfinance + indicators + chart PNGs).
# Idempotent — skip if already done.
!python scripts/build_dataset.py
!python scripts/validate_tasks.py

[1/6] Fetching OHLCV ...
  prices: 536 rows
[2/6] Computing indicators ...
  indicators: 536 rows
[3/6] Fetching peers + commodity ...
  peers: 374 rows
[4/6] Loading curated news/forums/macro ...
  news: 25 headlines, forums: 17 posts, macro: 12 events
[5/6] Rendering candlestick charts ...


In [16]:
from app.council.llm import TransformersLLMClient

# 4-bit BnB by default; drop load_in_4bit=False on L4/A100.
client = TransformersLLMClient.from_pretrained(
    "google/gemma-4-E4B-it",
    load_in_4bit=True,
)
print("Model loaded on:", client.model.device)

[transformers] Current model requires 6192 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [ ]:
# Pre-cache the 7 specialists' votes for every (task, step). Specialists
# are FROZEN, so this runs once and is reused across every GRPO step.
#
# Resumable: if the kernel disconnects mid-run, just re-run this cell —
# already-cached items are skipped in milliseconds.
import sys, time
from datetime import datetime
from tqdm import tqdm   # plain (text) tqdm — no ipywidgets dependency

from app.council.runner import Council
from app.core.environment import StockerEnv
from app.core.tasks import list_task_ids


def log(msg: str) -> None:
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)


log("Building plan ...")
council = Council(client=client, use_cache=True)

# 1) Flatten every (task, step, specialist) so tqdm has a real total.
plan = []
for task_id in list_task_ids():
    env = StockerEnv(task_id=task_id)
    obs = env.reset().observation
    while True:
        for sp in council.specialists:
            cache_path = council._vote_cache_path(sp, obs)
            plan.append((sp, obs, cache_path.exists()))
        result = env.step({"side": "hold", "quantity": 0})
        if result.done:
            break
        obs = result.observation

todo  = [(sp, obs) for sp, obs, hit in plan if not hit]
hits  = sum(1 for _, _, hit in plan if hit)
total = len(plan)
log(f"Plan: {total} (task,step,specialist) tuples — {hits} cached, {len(todo)} to compute.")
print("Per-specialist cache state:", flush=True)
for sp_cls in council.specialists:
    name = sp_cls.name
    n      = sum(1 for sp, _, _   in plan if sp.name == name)
    cached = sum(1 for sp, _, hit in plan if sp.name == name and hit)
    print(f"  {name:<18s} {cached:>3d}/{n:>3d} cached", flush=True)

if not todo:
    log("Nothing to do — all votes already cached.")
else:
    # 2) Run the misses. tqdm draws to stdout (text mode) and we ALSO emit a
    #    timestamped line every 25 items so even with output buffering the user
    #    can see progress.
    HEARTBEAT_EVERY = 25
    t0 = time.time()
    log(f"Computing {len(todo)} votes ... (heartbeat every {HEARTBEAT_EVERY})")
    pbar = tqdm(todo, desc="precaching", unit="vote", file=sys.stdout, dynamic_ncols=True)
    for i, (sp, obs) in enumerate(pbar, 1):
        council._cached_vote(sp, obs)
        pbar.set_postfix_str(f"{sp.name}@{obs.ticker}/{obs.date}")
        if i % HEARTBEAT_EVERY == 0 or i == len(todo):
            elapsed = time.time() - t0
            rate = i / max(elapsed, 1e-9)
            eta_s = (len(todo) - i) / max(rate, 1e-9)
            log(f"  {i}/{len(todo)} done  rate={rate:.2f} vote/s  ETA={eta_s/60:.1f} min")

    elapsed = time.time() - t0
    per_vote = elapsed / max(len(todo), 1)
    log(f"Done in {elapsed/60:.2f} min ({per_vote:.2f} s/vote). "
        f"Cache: {total} entries on disk.")

In [ ]:
# Pre-training baseline rollout (specialists from cache, moderator = base Gemma)
!python -m training.eval_rollout --out training/runs/eval_pre
!cat training/runs/eval_pre/summary.csv

In [ ]:
# GRPO on the moderator LoRA. Specialist votes are read from cache —
# only the moderator is re-rolled per training step, so this is fast.
#
# Knobs: --num-generations is K candidates per prompt (GRPO group size).
# On a free T4 keep batch_size=1, grad_accum=8.
!python -m training.train_grpo \
    --epochs 2 \
    --num-generations 8 \
    --batch-size 1 \
    --grad-accum 8 \
    --lora-rank 16 \
    --lr 5e-6

In [ ]:
# Post-training eval — load the trained adapter into the same client.
import glob, os
RUN_DIR = sorted(glob.glob("training/runs/grpo_*"))[-1]
LORA_DIR = os.path.join(RUN_DIR, "moderator-lora")
print("Using LoRA:", LORA_DIR)

# Attach adapter and re-run eval. The adapter name "moderator" matches what
# Moderator.decide() requests via extra_body={'lora_request': {'name': ...}}
from peft import PeftModel
client.model = PeftModel.from_pretrained(client.model, LORA_DIR, adapter_name="moderator")
client.moderator_lora = "moderator"

!python -m training.eval_rollout --moderator-lora moderator --out training/runs/eval_post
!cat training/runs/eval_post/summary.csv

In [ ]:
# Show training + eval curves inline
from IPython.display import Image, display
import os
for p in [f"{RUN_DIR}/loss.png", f"{RUN_DIR}/reward.png",
          "training/runs/eval_pre/reward_curve.png",
          "training/runs/eval_post/reward_curve.png",
          "training/runs/eval_pre/portfolio_curve.png",
          "training/runs/eval_post/portfolio_curve.png"]:
    if os.path.exists(p):
        print(p)
        display(Image(p))